In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Kernel_GLM_pseudo_labeling")
WORK_ROOT = Path("/content/Kernel_GLM_pseudo_labeling")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_DIR       = DRIVE_ROOT / "camelyon17_ckpts"
CKPT_DIR.mkdir(exist_ok=True)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from sklearn.metrics import log_loss, accuracy_score
from wilds import get_dataset
from PIL import Image
from pathlib import Path
import pickle
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
N_EPOCHS_IMPUTER   = 20    # train imputer to convergence
N_EPOCHS_CANDIDATE = 50    # candidate training epochs
BATCH_SIZE         = 256
LR                 = 4e-5
ACCUMULATION_STEPS = 1  # increase to 2 or 4 if OOM
WEIGHT_DECAY       = 0.01

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} — {torch.cuda.get_device_name(0)}")

Device: cuda — NVIDIA A100-SXM4-80GB


In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

# ── WILDS Wrapper ─────────────────────────────────────────────────────────────
class WILDSWrapper(torch.utils.data.Dataset):
    def __init__(self, wilds_subset, transform):
        self.subset    = wilds_subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        x, y, metadata = self.subset[idx]
        if not isinstance(x, Image.Image):
            x = transforms.ToPILImage()(x)
        if self.transform:
            x = self.transform(x)
        return x, y.item()

In [ ]:
# ── Load WILDS splits ─────────────────────────────────────────────────────────
print("Loading WILDS Camelyon17...")
dataset = get_dataset(dataset="camelyon17", download=True,
                      root_dir=str(WORK_ROOT))

In [ ]:
# D1: full train split — for candidate fine-tuning
# D2: id_val split    — for imputer fine-tuning and naive validation
# Target: test split  — hospital 4, unlabeled for pseudo, labeled for oracle
d1_data     = WILDSWrapper(dataset.get_subset("train",  transform=None),
                            train_transform)
d1_eval     = WILDSWrapper(dataset.get_subset("train",  transform=None),
                            eval_transform)   # eval version of D1 (no aug)
d2_train    = WILDSWrapper(dataset.get_subset("id_val", transform=None),
                            train_transform)
d2_eval     = WILDSWrapper(dataset.get_subset("id_val", transform=None),
                            eval_transform)
target_full = WILDSWrapper(dataset.get_subset("test",   transform=None),
                            eval_transform)

# Fixed split — same seed always gives same split
np.random.seed(42)
n_target = len(target_full)
idx_full = np.random.permutation(n_target)
n_sel    = n_target // 2

sel_idx  = idx_full[:n_sel]
test_idx = idx_full[n_sel:]

sel_data  = Subset(target_full, sel_idx)
test_data = Subset(target_full, test_idx)

y_sel  = np.array([target_full[i][1] for i in sel_idx])
y_test = np.array([target_full[i][1] for i in test_idx])

d1_loader        = DataLoader(d1_data,     batch_size=BATCH_SIZE,
                               shuffle=True,  num_workers=4, pin_memory=True)
d2_train_loader  = DataLoader(d2_train,    batch_size=BATCH_SIZE,
                               shuffle=True,  num_workers=4, pin_memory=True)
d2_eval_loader   = DataLoader(d2_eval,     batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=4, pin_memory=True)
sel_loader     = DataLoader(sel_data, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4, pin_memory=True)
test_loader    = DataLoader(test_data,batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
# ── Helper: get all probs and labels from a loader ────────────────────────────
@torch.no_grad()
def get_probs_labels(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        probs = torch.softmax(model(x.to(device)), dim=-1).cpu()
        all_probs.append(probs)
        all_labels.append(y)
    return (torch.cat(all_probs).numpy(),
            torch.cat(all_labels).numpy())

# ── Helper: build fresh DINOv2 + head ────────────────────────────────────────
def build_model(n_classes=2):
    backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    head     = nn.Linear(768, n_classes)
    return nn.Sequential(backbone, head).to(device)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Train imputer on D2 to convergence
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 1: Training imputer on D2 (id_val)...")
print("="*60)

imputer   = build_model()
optimizer = AdamW(imputer.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS_IMPUTER)
criterion = nn.CrossEntropyLoss()
d2_nlls = []  # before the loop

for epoch in range(N_EPOCHS_IMPUTER):
    imputer.train()
    train_loss = 0
    for x, y in d2_train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(imputer(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    # Monitor D2 train NLL
    probs_d2, labels_d2 = get_probs_labels(imputer, d2_eval_loader)
    d2_nll = log_loss(labels_d2, probs_d2)
    d2_nlls.append(d2_nll)
    print(f"  Imputer epoch {epoch+1:3d}/{N_EPOCHS_IMPUTER} — "
          f"train_loss={train_loss/len(d2_train_loader):.4f} | "
          f"D2 NLL={d2_nll:.4f}")

# Save imputer
torch.save(imputer.state_dict(), DRIVE_ROOT / "imputer.pt")
print("Imputer saved.")

np.savez(DRIVE_ROOT / "imputer_training.npz",
         d2_nlls=np.array(d2_nlls))  # add d2_nlls.append(d2_nll) inside imputer loop
print("Imputer training metrics saved.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Generate pseudo-labels (fixed for all candidate training)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2: Generating pseudo-labels on unlabeled target...")
print("="*60)

probs_target_pseudo, y_target = get_probs_labels(imputer, sel_loader)
pseudo_labels = probs_target_pseudo   # soft labels, shape (n_target, 2)

print(f"Pseudo-labels generated: {pseudo_labels.shape}")
print(f"Mean confidence: {pseudo_labels.max(axis=1).mean():.4f}")

# Save pseudo-labels and true target labels
np.savez(DRIVE_ROOT / "pseudo_labels.npz",
         pseudo_labels=pseudo_labels,
         y_target=y_target)
print("Pseudo-labels saved.")

# Free imputer from GPU
del imputer
torch.cuda.empty_cache()
print("Imputer deleted from GPU.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Fine-tune candidate model on D1, evaluate all signals each epoch
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 3: Fine-tuning candidate model on D1 (train)...")
print("="*60)

# Fresh initialization — independent from imputer
candidate  = build_model()
optimizer  = AdamW(candidate.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = CosineAnnealingLR(optimizer, T_max=N_EPOCHS_CANDIDATE)

pseudo_risks, naive_risks, oracle_risks = [], [], []
all_probs_target = []

for epoch in range(N_EPOCHS_CANDIDATE):
    # Train one epoch on D1
    candidate.train()
    train_loss = 0
    for x, y in d1_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(candidate(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    # Evaluate all three signals
    candidate.eval()
    with torch.no_grad():

        # Pseudo-target risk
        probs_target, _ = get_probs_labels(candidate, sel_loader)
        all_probs_target.append(probs_target)
        pseudo_risk = -np.mean(np.sum(
            pseudo_labels * np.log(probs_target + 1e-10), axis=1))

        # Naive: source val risk on D2
        probs_d2, labels_d2 = get_probs_labels(candidate, d2_eval_loader)
        naive_risk = log_loss(labels_d2, probs_d2)

        # Oracle: true target labels
        oracle_risk = log_loss(y_target, probs_target)

    pseudo_risks.append(pseudo_risk)
    naive_risks.append(naive_risk)
    oracle_risks.append(oracle_risk)

    # Save checkpoint to Drive
    torch.save(candidate.state_dict(),
               CKPT_DIR / f"epoch_{epoch+1:03d}.pt")

    print(f"Epoch {epoch+1:3d}/{N_EPOCHS_CANDIDATE} | "
          f"train={train_loss/len(d1_loader):.4f} | "
          f"naive={naive_risk:.4f} | "
          f"pseudo={pseudo_risk:.4f} | "
          f"oracle={oracle_risk:.4f}")

# Save risk curves immediately
np.savez(DRIVE_ROOT / "early_stopping_risks.npz",
         pseudo_risks=np.array(pseudo_risks),
         naive_risks=np.array(naive_risks),
         oracle_risks=np.array(oracle_risks))
print("\nRisk curves saved.")

# After the loop:
np.savez(DRIVE_ROOT / "all_epoch_probs.npz",
         all_probs_target=np.array(all_probs_target),
         y_target=y_target)
print("Per-epoch target probabilities saved.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Select best epoch and evaluate
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 4: Selecting best epoch and evaluating...")
print("="*60)

epoch_naive  = int(np.argmin(naive_risks))
epoch_pseudo = int(np.argmin(pseudo_risks))
epoch_oracle = int(np.argmin(oracle_risks))

print(f"Selected epochs:")
print(f"  Naive:  {epoch_naive  + 1}")
print(f"  Pseudo: {epoch_pseudo + 1}")
print(f"  Oracle: {epoch_oracle + 1}")

results = {}
for name, epoch in [("naive",  epoch_naive),
                     ("pseudo", epoch_pseudo),
                     ("oracle", epoch_oracle)]:
    candidate.load_state_dict(
        torch.load(CKPT_DIR / f"epoch_{epoch+1:03d}.pt",
                   map_location=device))
    probs, labels = get_probs_labels(candidate, test_loader)
    results[name] = {
        "epoch":    epoch + 1,
        "nll":      log_loss(labels, probs),
        "accuracy": accuracy_score(labels, probs.argmax(axis=1)),
    }

print(f"\n{'Method':<10} {'Epoch':>7} {'NLL':>8} {'Accuracy':>10}")
print("-" * 40)
for method in ["naive", "pseudo", "oracle"]:
    r = results[method]
    print(f"{method:<10} {r['epoch']:>7} {r['nll']:>8.4f} "
          f"{r['accuracy']:>10.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Save everything
# ══════════════════════════════════════════════════════════════════════════════
with open(DRIVE_ROOT / "early_stopping_results.pkl", "wb") as f:
    pickle.dump({
        "results":           results,
        "pseudo_risks":      pseudo_risks,
        "naive_risks":       naive_risks,
        "oracle_risks":      oracle_risks,
        "d2_nlls":           d2_nlls,
        "epoch_naive":       epoch_naive,
        "epoch_pseudo":      epoch_pseudo,
        "epoch_oracle":      epoch_oracle,
        "all_probs_target":  all_probs_target,  # ADD
        "pseudo_labels":     pseudo_labels,      # ADD
        "y_target":          y_target,           # ADD
        "config": {
            "n_epochs_imputer":   N_EPOCHS_IMPUTER,
            "n_epochs_candidate": N_EPOCHS_CANDIDATE,
            "lr":                 LR,
            "batch_size":         BATCH_SIZE,
            "weight_decay":       WEIGHT_DECAY,
        }
    }, f)
print("All results saved to Drive.")

# ══════════════════════════════════════════════════════════════════════════════
# Plot
# ══════════════════════════════════════════════════════════════════════════════
epochs = np.arange(1, N_EPOCHS_CANDIDATE + 1)
colors = {"naive": "blue", "pseudo": "orange", "oracle": "green"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: risk curves
ax = axes[0]
for method, risks in [("naive",  naive_risks),
                       ("pseudo", pseudo_risks),
                       ("oracle", oracle_risks)]:
    ax.plot(epochs, risks, color=colors[method], label=method)
    ax.axvline(np.argmin(risks) + 1, color=colors[method],
               linestyle="--", alpha=0.7)
ax.set_xlabel("Epoch")
ax.set_ylabel("Risk")
ax.set_title("Early stopping — risk curves per method")
ax.legend()

# Right: final NLL bar chart
ax = axes[1]
methods = ["naive", "pseudo", "oracle"]
nlls    = [results[m]["nll"] for m in methods]
accs    = [results[m]["accuracy"] for m in methods]
x       = np.arange(len(methods))
bars    = ax.bar(methods, nlls,
                  color=[colors[m] for m in methods], alpha=0.7)
for i, (nll, acc) in enumerate(zip(nlls, accs)):
    ax.text(i, nll + 0.002, f"NLL={nll:.4f}\nAcc={acc:.4f}",
            ha="center", fontsize=9)
ax.set_ylabel("Target NLL")
ax.set_title("Final target performance by method")

plt.suptitle("Camelyon17 — DINOv2 fine-tuning early stopping",
             fontsize=13)
plt.tight_layout()
plt.savefig(DRIVE_ROOT / "early_stopping_plot.png",
            dpi=150, bbox_inches="tight")
plt.show()